# ⚠️ EJECUTAR EN KAGGLE (T4) — no ejecutar en local

# 05 — Entrenamiento QLoRA de SkillGen (experimento E1)

Este es el notebook de **producción** del entrenamiento real: QLoRA sobre Qwen2.5-7B-Instruct (base **provisional**, PLAN.md §15) con la receta E1 de §18. Está pensado para ejecutarse **íntegramente en Kaggle**, no en la VM local (sin GPU, este notebook no puede entrenar nada; en local solo existe el SMOKE de CPU con el modelo de 0.5B, `configs/smoke_cpu.yaml`, que prueba el pipeline pero **no** es E1).

## Prerrequisitos (gates antes de ejecutar)

1. **Cuenta de Kaggle verificada** (verificación telefónica hecha) y sesión con **GPU T4 activada** (Settings → Accelerator → GPU T4). **No uses P100**: los kernels 4-bit actuales de bitsandbytes/Unsloth ya no compilan para Pascal (sm_60) y el run muere con `CUDA error: no kernel image is available` (§16).
2. **El repo subido a `/kaggle/working`**: como Kaggle Dataset versionado o clonado al inicio de la sesión, con los splits congelados en `data/splits/` (⚠️ el test sigue congelado: jamás se entrena con él; este notebook ni lo lee).
3. **Duración esperada** (cualitativa): E1 son ~582 ejemplos filtrados × 2 epochs con batch efectivo 16 → del orden de **~73 steps** de entrenamiento más evaluaciones periódicas; en una T4 es un trabajo de **horas, no de minutos** — cabe en una sesión de Kaggle (~9–12 h) y en el presupuesto semanal (~30 h), pero no dejes la sesión sin vigilar la primera vez.
4. **Gates que no se negocian** (§21): el template de chat es el nativo del modelo; EOS al final de cada ejemplo; y la **verificación de masking** de la celda 4 se lee con los ojos antes de entrenar. Un loss que baja suave NO demuestra que el template esté bien.
5. **Decisión de longitud (Fase 0)**: `max_seq_length = 8192` con filtro (nunca truncar): 218 de 800 registros de train quedan fuera, 582 entran enteros. Esto **sustituye** al 4096 provisional de §18.
6. **Tracking** (§28): si defines `WANDB_API_KEY` en los Secrets de Kaggle, el run reporta a Weights & Biases; sin la key, `report_to=[]` y el run es offline sin error.

## Celda 1 — Instalación (solo Kaggle)

`bitsandbytes` y `unsloth` **no** son dependencias del proyecto en local, a propósito:

- **bitsandbytes** compila kernels CUDA: no puede ni importarse en la VM de CPU.
- **unsloth** fija su propia pila de torch/xformers, que pelearía con el pin de torch-CPU que usa el desarrollo local (`pyproject.toml`).

En Kaggle la pila CUDA ya existe, así que aquí se instalan en celda, junto con las dependencias normales del pipeline de entrenamiento (trl/peft/accelerate/datasets).

In [ ]:
# SOLO KAGGLE: unsloth fija su propia pila CUDA; bitsandbytes necesita GPU.
# En local estas instalaciones están prohibidas por diseño (ver la explicación arriba).
!pip install -q unsloth
!pip install -q trl peft accelerate datasets pyyaml
print("Instalación completa. Si unsloth cambió la versión de torch, reinicia el kernel y continúa desde la celda 2.")

## Celda 2 — Verificación del entorno

La misma lógica del notebook `00_environment.ipynb`, en forma de asserts duros: si la GPU no es la esperada, este notebook **se detiene aquí** en vez de entrenar mal durante horas. La regla de reserva (§16 y notebook 00): si VRAM < 16 GB o compute capability < 7.0 → `max_seq_length = 2048`, batch por dispositivo 1 y más acumulación (habría que editar la config, no seguir con la de E1).

In [ ]:
import sys

assert sys.version_info >= (3, 12), "SkillGen requiere Python >= 3.12"

import torch

print("torch:", torch.__version__)
assert torch.cuda.is_available(), "Sin GPU no se entrena (§16): activa la T4 en Settings → Accelerator."

props = torch.cuda.get_device_properties(0)
cc = props.major + props.minor / 10
vram_gb = props.total_memory / (1024 ** 3)
print(f"GPU: {props.name} | compute capability: {cc} | VRAM: {vram_gb:.1f} GB")

# Los kernels 4-bit de bitsandbytes/Unsloth exigen sm_70+ (nada de P100/Pascal).
assert cc >= 7.0, f"Compute capability {cc} < 7.0: cambia a una T4."
if vram_gb < 16:
    print("WARN: VRAM < 16 GB → aplica la regla de reserva de §16 (max_seq_length 2048, batch 1, más acumulación) y NO uses configs/qlora_e1.yaml tal cual.")
else:
    print("OK: entorno listo para E1 (max_seq_length 8192).")

## Celda 3 — 🚦 GATE de masking (§21): PARA Y LEE

Antes de entrenar, construimos **un** ejemplo real con `prepare.py` — el mismo código que certifican los tests — y decodificamos los tokens que NO están enmascarados.

**Lo que tienes que ver impreso:** únicamente el JSON del blueprint del registro (el turno del assistant) seguido de `<|im_end|>`. Nada del system prompt, nada de la instrucción del usuario.

Si ves texto del prompt entre lo desenmascarado, el template está mal y entrenar sería quemar horas de GPU para aprender nada: el masking defectuoso enmascara todo en silencio y el loss sale 0.0. **No sigas a la celda de entrenamiento sin leer esta salida.**

In [ ]:
import sys
from pathlib import Path

# El repo debe estar en /kaggle/working (Dataset o clon). Ajusta si lo subiste a otra ruta.
REPO = Path("/kaggle/working/skillgen")
sys.path.insert(0, str(REPO / "src"))

from transformers import AutoTokenizer

from skillgen.evaluation.baseline import load_split_records
from skillgen.training.config import load_config
from skillgen.training.prepare import build_example, verify_masking

config = load_config(REPO / "configs" / "qlora_e1.yaml")
tokenizer = AutoTokenizer.from_pretrained(config["model"]["name"])

# Un registro CORTO y dentro del cap de 8192: el primer registro del split es
# el gigante de ~1.9M tokens (ver EDA de la Fase 0) y su blueprint llenaría
# la pantalla; para LEER el gate queremos un ejemplo completo y legible.
records = load_split_records("train", splits_dir=REPO / "data" / "splits")
record = next(
    r for r in records
    if len(r["instruction"]) + len(r["expected_blueprint"]) < 20000
)
example = build_example(record, tokenizer, max_seq_length=config["data"]["max_seq_length"])
decoded = verify_masking(example, tokenizer, record)  # lanza excepción si el masking falla

print("=== TEXTO DESENMASCARADO (debe ser SOLO el blueprint + <|im_end|>) ===")
print(decoded)
print("=== FIN ===")
print(f"Tokens totales: {len(example['input_ids'])} | enmascarados (-100): {sum(1 for lab in example['labels'] if lab == -100)}")


## Celda 4 — Entrenamiento E1

Ejecuta el entry point real del repo (`train.py`), el mismo que corre el SMOKE en local: carga la config, construye el dataset pre-tokenizado con los labels de `prepare.py`, **repite el gate de masking e imprime el texto desenmascarado** (vuelve a leerlo en el log), entrena con early stopping sobre val loss y guarda el adapter (no el modelo completo, §27) en `experiments/e1/`.

La receta (§18, con la longitud corregida por la Fase 0): NF4 + doble cuantización, cómputo fp16 (la T4 no tiene bf16), LoRA r16/α32/dropout 0 sobre todos los lineales, lr 2e-4 cosine, warmup 3%, batch efectivo 16 (2 × acumulación 8), 2 epochs, seed 42, checkpoints cada 50 steps con el mejor por val loss.

> El modelo base cargado es el checkpoint pre-cuantizado `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (eso pone `model.name` en la config; la base sigue siendo **provisional** hasta verificar los modelos 2026, §15).

In [ ]:
# Entrenamiento E1 real (7B QLoRA). Tarda horas en la T4: no cierres la sesión.
!cd /kaggle/working/skillgen && python -m skillgen.training.train --config configs/qlora_e1.yaml

## Celda 5 — Evaluación del modelo entrenado y rescate del adapter

Entrenar no basta: hay que puntuar el modelo ajustado con **el mismo harness** de los baselines para que la comparación E0 vs E1 sea honesta (mismos prompts §8, mismo split de val, misma escalera de 5 capas). El flujo:

1. Genera predicciones con el adapter cargado sobre el modelo base usando el pipeline de la Fase 2 (`scripts/run_baseline.py` es para el modelo *sin* entrenar; para el ajustado, carga base + adapter con PEFT y genera con la misma función `generate_predictions` de `skillgen.evaluation.baseline`, guardando en `experiments/e1/predictions.jsonl`).
2. Puntúa con `score_predictions` (reutiliza `scripts/eval_harness.py`; L4 seguirá «skipped» sin `OPENROUTER_API_KEY` y el score se renormaliza sobre 0.85 — dilo así al reportar).
3. **Rescata el adapter antes de que muera la sesión**: la salida de Kaggle solo persiste lo que quede en `/kaggle/working` al guardar la versión del notebook. El adapter son unos pocos cientos de MB como máximo (adapter + tokenizer + checkpoints); el modelo completo de 7B **no** se guarda (§27: se versionan adapters).

Comandos de referencia para la copia de seguridad:

In [ ]:
# Referencia (ejecutar y adaptar en la sesión de Kaggle tras entrenar):
# 1) ¿Qué hay que rescatar?
!ls -lh /kaggle/working/skillgen/experiments/e1/
# Debe contener: adapter_model.safetensors, adapter_config.json, tokenizer*,
# y checkpoint-*/ con el mejor checkpoint por val loss.

# 2) Comprime SOLO el adapter final (sin checkpoints intermedios) para descargarlo:
!cd /kaggle/working/skillgen/experiments/e1 && tar -czf /kaggle/working/e1_adapter.tar.gz adapter_model.safetensors adapter_config.json tokenizer.json tokenizer_config.json special_tokens_map.json 2>/dev/null || echo "Ajusta la lista de archivos a lo que haya en experiments/e1/"
!ls -lh /kaggle/working/e1_adapter.tar.gz

## Qué sigue después de E1

- **Ablaciones (§19), un cambio por run**, cada una con su config ya creada en `configs/`: E2 sin masking, E3 epochs 1/3, E4 rank 8/32, E5 solo atención, E6 datos 250/500 (ojo: del pool **filtrado** de 582, no de los 800 originales). Mismo seed, mismos prompts, mismo test congelado: así cada diferencia es atribuible a su único cambio.
- **Fase 5 (export)**: merge del adapter a 16 bits → GGUF `q4_k_m` → Ollama, verificando que el template de chat del export es idéntico al de entrenamiento (el mismatch train/inferencia es la causa nº 1 de salidas basura post-export, §21).
- Los hallazgos negativos también se reportan: si una ablación no mejora, se escribe tal cual.